# Проект 1. Окупилось ли повышение цен в марте 2026

**Бизнес-вопрос.** С 1 марта 2026 ПРАЙМ поднял цены: "Базовый" с 199 до 249 руб., "Расширенный" с 399 до 449 руб.
Принесло ли это больше денег, или часть выигрыша съел отток? Что делать с ценами дальше?

**Гипотезы**, которые проверяем в ноутбуке:

1. Выручка после повышения выросла за счёт цены, а не за счёт базы подписок.
2. После повышения вырос отток: клиенты ушли из-за цены.
3. Сильнее ушли клиенты дешёвого тарифа или отдельных каналов привлечения.

**Договорённости расчёта** (пишем их в подписи к каждому числу):

- время и месяцы считаем по **UTC**, так хранит база;
- выручка это сумма **успешных платежей** (статус `status = 'success'`), возвраты (`refunded`) и неудачные (`failed`) не берём;
- выручку берём из таблицы платежей (`payments`) и **не соединяем** с участниками подписок (`subscription_members`): так нет двойного счёта из-за шеринга;
- месяцы разной длины сравниваем **в пересчёте на день**;
- клиентов без канала привлечения показываем отдельной строкой "без канала".

## 0. Подключение

Функция запроса (`q`) отправляет SQL в базу и возвращает таблицу pandas. Пароля в ноутбуке нет:
его хранит файл `.env`, а читает функция подключения (`get_engine`) из `src/prime/config.py`.

In [ ]:
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

# Наши функции из репозитория: подключение, проверки, метрики, разложение.
try:
    from prime.config import get_engine
    from prime.checks import find_duplicate_payments, find_missing_channel
    from prime.metrics import active_on, churn_30d, churn_rate, daily_churn
    from prime.factors import chain, contrib, waterfall
except ModuleNotFoundError:
    # Понятная ошибка вместо длинного трейсбека: ноутбук открыт не в окружении проекта.
    raise ModuleNotFoundError(
        "Пакет prime не найден. Запустите Jupyter из корня репозитория командой: uv run jupyter lab\n"
        f"Сейчас Python отсюда: {sys.executable}"
    )

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть DataFrame. Соединение закрываем сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они одинаковые у всех, кто запускает ноутбук.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        # Сервер общий на всю группу, лишние соединения не держим.
        engine.dispose()


# Тысячи через пробел в выводе таблиц: так числа читать легче.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))

print("pandas", pd.__version__)
print("подключились как:", q("select current_user as login").iloc[0, 0])

**Вывод.** Подключение работает: pandas 3.0.5, вошли под своим логином. Функции из `src/prime` импортировались.

**Следующий шаг: приёмка данных.** Ошибки pandas не падают, а тихо меняют числа (Л3). Поэтому до любого расчёта
проверяем таблицы семью вопросами приёмки.

## 1. Приёмка данных

Задаём таблицам семь вопросов приёмки из Л3: сколько строк и сколько ждали, какие типы, сколько весит,
одна строка это кто, где пропуски, нет ли невозможных значений, нет ли дыр по датам.

Нужны три таблицы:

| Таблица | Зерно (одна строка это) | Зачем нам |
|---|---|---|
| справочник тарифов (`tariffs`) | цена тарифа в интервале дат | узнать, когда и насколько подняли цены |
| платежи (`payments`) | одна попытка списания | выручка и число платежей |
| подписки (`subscriptions`) | одна подписка | активная база, отток, канал владельца |

### 1.1 Справочник цен

У тарифа несколько строк: цена меняется, и у каждой строки свой интервал от даты начала (`valid_from`)
до даты конца (`valid_to`). На семинаре 4 в нём нашлась строка, которая перекрывает соседние.

In [ ]:
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

print(tariffs.dtypes)
tariffs

**Вывод.** В справочнике 5 строк на 2 тарифа. Цена в месяц (`monthly_fee`) приехала числом, а даты начала и конца
(`valid_from`, `valid_to`) приехали объектами (`object`), а не временем: сравнивать их с моментом платежа напрямую нельзя.
Подозрительна строка "Расширенного" за 429 руб. с 01.02.2026 по 30.06.2026: она залезает и на старую, и на новую цену.

**Следующий шаг: найти пересечения интервалов кодом.** На 5 строках это видно глазами, но вывод должен опираться
на проверку, которая сработает на любом справочнике (Л4, слайд 14).

In [ ]:
# Ищем пересечения интервалов самосоединением (Л4, слайд 14).
# Соединяем справочник сам с собой по тарифу: получаем все пары строк одного тарифа.
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))

overlap = pairs[
    (pairs["row_id_a"] < pairs["row_id_b"])               # без этого каждая строка "пересечётся" сама с собой
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])      # два интервала пересекаются, если каждый
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])      # начинается не позже конца другого
]

print("пересекающихся пар:", len(overlap))
overlap[["tariff_id", "monthly_fee_a", "valid_from_a", "valid_to_a", "monthly_fee_b", "valid_from_b", "valid_to_b"]]

**Вывод.** Пересекающихся пар 2, и в обеих участвует строка 429 руб.: она перекрывает цену 399 руб. (февраль 2026)
и цену 449 руб. (с марта). С таким справочником каждый платёж "Расширенного" за февраль-июнь найдёт две цены.

**Следующий шаг: загрузить платежи.** Какую строку считать ошибкой, решаем не глазами, а по данным:
в разделе 1.3 посмотрим, какие суммы реально списывались.

### 1.2 Платежи

Берём период с декабря 2025 по июнь 2026: три месяца до повышения (декабрь-февраль) и четыре после (март-июнь).
Эффект цены меряем не за один март, а сравнением двух окон по 3 месяца: декабрь-февраль против марта-мая.
Один месяц может оказаться случайным, а за 3 месяца каждая подписка успевает продлиться по новой цене трижды.
Июнь оставляем для проверки, что эффект держится и дальше.
Июль и август не берём: в конце августа случился обвал партнёрского канала (`partner_telecom`, Л6), он смешал бы два разных события.

Колонки выбираем в SQL, а не "выгружу всё, потом посмотрю": так таблица весит меньше (Л3, слайд 11).
Тариф берём из подписки: у платежа своего тарифа нет.

In [ ]:
PERIOD = ("2025-12-01", "2026-07-01")   # [начало, конец): правый конец не включён

# Контрольное число строк считаем в базе ДО выгрузки: потом сверим с тем, что приехало.
expected_rows = q("""
    select count(*) from prime.payments
    where paid_at >= :a and paid_at < :b
""", a=PERIOD[0], b=PERIOD[1]).iloc[0, 0]

pay = q("""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, s.tariff_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.paid_at >= :a and p.paid_at < :b
""", a=PERIOD[0], b=PERIOD[1])

# Вопрос 1. Сколько строк и сколько ждали. inner join с подписками не должен потерять ни одной строки:
# у каждого платежа есть подписка.
print("1. строк:", len(pay), " ждали:", expected_rows, "->", "ok" if len(pay) == expected_rows else "ВНИМАНИЕ")
# Вопрос 2. Какие типы выбрал за нас pandas.
print("\n2. типы:"); print(pay.dtypes)
# Вопрос 3. Сколько весит. deep=True обязателен: без него pandas не считает сами строки текста.
print(f"\n3. вес: {pay.memory_usage(deep=True).sum() / 1e6:.1f} МБ")

**Вывод.** Приехало 1 268 997 строк, ровно столько же, сколько в базе: соединение с подписками (`join`) ничего не потеряло.
Время платежа (`paid_at`) пришло с поясом UTC, сумма (`amount`) пришла дробным числом (`float64`): для сумм по выручке
этого хватает (Л3, слайд 12). Идентификаторы (`payment_id`, `subscription_id`) пришли объектами (`object`). Таблица весит 284 МБ.

**Следующий шаг: вопросы 4-6.** Строк столько, сколько ждали, но ещё не ясно, одна ли строка это один платёж,
есть ли пропуски и невозможные значения.

In [ ]:
# Вопрос 4. Одна строка это один платёж? Тогда payment_id уникален.
print("4. payment_id уникален:", pay["payment_id"].is_unique)

# Двойные списания (С3): та же подписка и сумма меньше чем через 20 минут. Ищем среди успешных.
ok = pay[pay["status"] == "success"]
dupes = find_duplicate_payments(ok)
print(f"   двойных списаний: {len(dupes)} на сумму {dupes['amount'].sum():,.0f} руб.".replace(",", " "))

# Вопрос 5. Где пропуски.
print("\n5. пропуски по колонкам:"); print(pay.isna().sum())

# Вопрос 6. Нет ли невозможных значений.
print("\n6. статусы:"); print(pay["status"].value_counts(dropna=False))
print("   сумма <= 0:", (pay["amount"] <= 0).sum())

**Вывод.** Идентификатор платежа (`payment_id`) уникален: одна строка это одна попытка списания. Пропусков нет ни в одной колонке.
Статусы три: успешные (`success`) 1 211 707, неудачные (`failed`) 41 944, возвраты (`refunded`) 15 346. Сумм ноль и меньше нет.
Функция поиска двойных списаний (`find_duplicate_payments`) нашла 1 189 повторов на 362 961 руб.

**Следующий шаг: проверить, что это действительно двойные списания.** Если это обычные продления,
исключать их нельзя: мы бы занизили выручку. Поэтому смотрим, сколько времени проходит между платежами.

#### Почему мы считаем их двойными списаниями

Подписка списывается раз в 30 дней. Значит, между двумя успешными платежами одной подписки на одну сумму
в норме проходит около 30 дней. Если проходят минуты, это не продление, а повтор того же списания.
Проверяем это на наших данных, а не верим порогу с семинара на слово.

In [ ]:
# Для каждого успешного платежа: сколько прошло с предыдущего платежа той же подписки на ту же сумму.
s = ok.sort_values(["subscription_id", "amount", "paid_at"])
gap = s.groupby(["subscription_id", "amount"])["paid_at"].diff()   # у первого платежа подписки будет NaT

# Раскладываем промежутки по корзинам: где живут "нормальные" платежи, а где повторы.
bins = pd.to_timedelta(["0s", "1min", "20min", "1D", "25D", "35D", "3650D"])
labels = ["меньше 1 минуты", "1-20 минут", "20 минут - 1 день", "1-25 дней", "25-35 дней (продление)", "больше 35 дней"]
print("Промежуток до предыдущего такого же платежа:")
print(pd.cut(gap.dropna(), bins=bins, labels=labels, right=False).value_counts(sort=False))

# Анализ чувствительности (С3): если ответ не меняется в широком диапазоне порога,
# значит он не зависит от нашего выбора "20 минут".
print("\nПорог -> сколько повторов:")
for m in (1, 5, 15, 20, 60, 180, 1440, 7 * 1440):
    print(f"{m:>6} мин -> {(gap < pd.Timedelta(minutes=m)).sum()}")

**Вывод.** Промежутки делятся на две группы без середины: 1 189 повторов через 0-15 минут и 776 726 продлений через 25-35 дней.
Между 20 минутами и 25 днями нет ни одного платежа. Порог можно ставить от 15 минут до недели: число повторов одно и то же, 1 189.
Значит, ответ не зависит от нашего выбора порога.

**Следующий шаг: посмотреть на повторы вблизи** и проверить, вернули ли за них деньги. Если вернули,
то в выручке их и так нет; если нет, их надо убрать самим.

In [ ]:
# Как выглядят повторы вблизи: через сколько минут после первого платежа и сколько подписок задето.
is_dup = gap < pd.Timedelta(minutes=20)
print("медиана промежутка у повторов:", gap[is_dup].median())
print("самый длинный промежуток у повторов:", gap[is_dup].max())
print("подписок с повтором:", s.loc[is_dup, "subscription_id"].nunique())

# Вернули ли эти деньги? Ищем возврат (refunded) по той же подписке в течение 7 дней после повтора.
refunds = pay.loc[pay["status"] == "refunded", ["subscription_id", "paid_at"]]
d = s.loc[is_dup, ["subscription_id", "paid_at"]].merge(refunds, on="subscription_id", suffixes=("", "_ref"))
d = d[(d["paid_at_ref"] >= d["paid_at"]) & (d["paid_at_ref"] < d["paid_at"] + pd.Timedelta(days=7))]
print("повторов, по которым был возврат в течение 7 дней:", d["paid_at"].nunique(), "из", is_dup.sum())

# Пример одной подписки с повтором: все её успешные платежи в периоде.
example = s.loc[is_dup, "subscription_id"].iloc[0]
s.loc[s["subscription_id"] == example, ["paid_at", "amount", "status"]]

**Вывод.** Медиана промежутка у повторов 7 минут 51 секунда, самый длинный 14 минут 59 секунд: так выглядит повтор запроса
платёжным шлюзом, а не решение клиента. Задето 1 185 подписок. Возвратов по повторам нет ни одного: деньги взяли дважды
и не вернули. В примере подписка платит раз в месяц, а 28 апреля платит дважды с разницей 5 минут.

**Решение: исключаем повторы из выручки** (раздел 1.5). Для бизнеса это отдельная находка: по 1 185 подпискам надо вернуть лишние списания.

**Следующий шаг: вопрос 7, дыры по дням.** От полноты дней зависит, можно ли сравнивать месяцы между собой.

In [ ]:
# Вопрос 7. Какой период покрыт и нет ли дыр по дням.
# Сравниваем дни, которые есть в данных, с полным календарём периода.
days_in_data = pd.DatetimeIndex(pay["paid_at"].dt.tz_localize(None).dt.normalize().unique())
all_days = pd.date_range(PERIOD[0], PERIOD[1], freq="D", inclusive="left")   # inclusive="left": без 1 июля
missing = all_days.difference(days_in_data)

print("7. первый и последний платёж:", pay["paid_at"].min(), "...", pay["paid_at"].max())
print("   дней в календаре:", len(all_days), " дней с платежами:", len(days_in_data))
print("   дни без единого платежа:", [d.strftime("%Y-%m-%d") for d in missing])

**Вывод.** Период покрыт с 1 декабря 2025 по 30 июня 2026: 212 дней в календаре, платежи есть в 210.
Нет ни одного платежа 14 и 15 апреля 2026: это дыра в выгрузке, известная с Л5. Апрель будем считать
в пересчёте на день только по 28 дням с данными. Февраль и март дыра не задевает.

**Следующий шаг: посмотреть, какие суммы реально списывались по месяцам.** Это ответит на два вопроса сразу:
когда новая цена дошла до клиентов и настоящая ли строка 429 руб.

### 1.3 Какие суммы реально списывались

Цены в справочнике поменялись 1 марта. Но это не значит, что 1 марта все стали платить по-новому:
новая цена могла включаться только при следующем продлении.

In [ ]:
# Месяц платежа по UTC. tz_localize(None) снимает пояс перед to_period: иначе pandas предупреждает,
# что пояс потеряется (Л4, слайд 23). Время при этом не сдвигается: остаются часы UTC.
ok = pay[pay["status"] == "success"].copy()
ok["month"] = ok["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Строки: тариф и сумма платежа. Колонки: месяцы слева направо. В клетке: сколько платежей.
amounts = ok.groupby(["tariff_id", "amount", "month"]).size().unstack("month", fill_value=0).sort_index(axis=1)
amounts

**Вывод.** Новая цена включилась для всех сразу 1 марта: до марта нет ни одного платежа по 249 и 449 руб., с марта ни одного
по 199 и 399 руб. Платежей на 429 руб. нет ни одного, значит строка 429 руб. в справочнике это ошибка ввода, а не настоящая цена.

Отсюда важное следствие для метода: граница чёткая, переходного периода нет. Окно "до" (декабрь-февраль) целиком по старым ценам, окно "после" (март-май) целиком по новым, и цены в окнах не смешиваются.

**Следующий шаг: подписки.** Для активной базы и оттока нужны даты начала и конца подписок и канал владельца.

### 1.4 Подписки

Канал привлечения (`acquisition_channel`) берём у **владельца** подписки. У части клиентов канала нет: считаем их
с параметром "не выбрасывать пропуски" (`dropna=False`), чтобы они не пропали молча (Л3, слайд 25).

In [ ]:
subs = q("""
    select s.subscription_id, s.tariff_id, s.started_at, s.ended_at, s.is_trial,
           c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
""")

expected_subs = q("select count(*) from prime.subscriptions").iloc[0, 0]

print("строк:", len(subs), " ждали:", expected_subs, "->", "ok" if len(subs) == expected_subs else "ВНИМАНИЕ")
print("subscription_id уникален:", subs["subscription_id"].is_unique)
print("\nтипы:"); print(subs.dtypes)
print("\nпропуски:"); print(subs.isna().sum())
# Инвариант двух колонок: подписка не может закончиться раньше, чем началась.
print("\nзакончились раньше начала:", (subs["ended_at"] < subs["started_at"]).sum())
print("\nканал владельца, вместе с пропусками:")
print(subs["acquisition_channel"].value_counts(dropna=False))

**Вывод.** Подписок 333 905, ровно как в базе, идентификатор подписки (`subscription_id`) уникален.
Дата конца (`ended_at`) пуста у 167 978 подписок: это штатно, они ещё действуют. Подписок, которые закончились раньше,
чем начались, нет. Канала нет у 6 736 подписок: без параметра `dropna=False` они бы молча выпали из разбивки по каналам.

**Следующий шаг: подвести итог приёмки и собрать чистые таблицы**, чтобы дальше все расчёты шли от одних и тех же данных.

### 1.5 Итог приёмки и чистые таблицы

| Что нашли | Что делаем |
|---|---|
| Строка 429 руб. в справочнике перекрывает две соседние, платежей на 429 руб. нет ни одного | убираем строку из справочника |
| Новая цена включилась для всех 1 марта 2026: до марта нет ни одного платежа по новой цене, с марта ни одного по старой | сравниваем 3 месяца до (декабрь-февраль) и 3 месяца после (март-май) |
| 1 189 двойных списаний (повтор через 0-15 минут), возвратов по ним нет | исключаем из выручки |
| 14 и 15 апреля 2026 нет ни одного платежа: дыра в выгрузке | апрель считаем в день только по дням с данными |
| Неудачные (`failed`) и возвраты (`refunded`) | в выручку не берём |
| 6 736 подписок без канала владельца | показываем строкой "без канала" |

In [ ]:
# Справочник без кривой строки 429 руб.: ни одного платежа на 429 руб. нет (раздел 1.3), значит это ошибка ввода.
tariffs_fixed = tariffs[tariffs["monthly_fee"] != 429].copy()

# Проверяем, что пересечений больше нет. Проверка, которая может упасть, и есть настоящая проверка.
a = tariffs_fixed.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))
n_overlap = ((pairs["row_id_a"] < pairs["row_id_b"])
             & (pairs["valid_from_a"] <= pairs["valid_to_b"])
             & (pairs["valid_from_b"] <= pairs["valid_to_a"])).sum()
assert n_overlap == 0, "в справочнике остались пересечения"
print(tariffs_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))

**Вывод.** В чистом справочнике (`tariffs_fixed`) 4 строки, пересечений нет, проверка (`assert`) прошла:
199 и 399 руб. до 28.02.2026, 249 и 449 руб. с 01.03.2026. Повышение: "Базовый" +25,1%, "Расширенный" +12,5%.

**Следующий шаг: убрать двойные списания** и собрать рабочую таблицу платежей для всех следующих расчётов.

In [ ]:
success = pay[pay["status"] == "success"]   # деньги приносят только успешные платежи

# Исключили платежи, чтобы не вести двойной счёт по двойным списаниям за 1 подписку в течение 20 минут.
# (Нормально, если платёж повторяется через 25+ дней. Подтверждено данными: ячейка с корзинами промежутков выше.)
dupes = find_duplicate_payments(success, gap_minutes=20)   # функция возвращает ВТОРОЙ платёж пары, первый остаётся
paid = success[~success.index.isin(dupes.index)].copy()

# Месяц платежа по UTC: понадобится на шаге 2.
paid["month"] = paid["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Контроль: убрали ровно столько строк, сколько нашли дублей, и ни одной лишней.
assert len(paid) == len(success) - len(dupes), "убрали не те строки"
print(f"успешных платежей: {len(success):>9}")
print(f"из них дублей:     {len(dupes):>9}")
print(f"осталось (paid):   {len(paid):>9}")
print(f"выручка до чистки:   {success['amount'].sum():>15,.2f} руб.".replace(",", " "))
print(f"выручка после:       {paid['amount'].sum():>15,.2f} руб.".replace(",", " "))

**Вывод.** Из 1 211 707 успешных платежей убрали 1 189 повторов, в рабочей таблице (`paid`) осталось 1 210 518.
Выручка за 7 месяцев: 368 919 493 руб. до чистки и 368 556 532 руб. после, разница 362 961 руб.
Проверка (`assert`) подтвердила, что убрали ровно повторы и ни одной лишней строки.

**Следующий шаг: метрики по месяцам.** Данные чистые, теперь можно собирать таблицу, из которой будем раскладывать выручку.

## 2. Метрики по месяцам и зерно расчёта

**Зерно рабочей таблицы: одна строка это месяц × тариф × канал владельца.** Почему именно так:

- **месяц**: подписка списывается раз в 30 дней, значит месяц это естественный период, в котором каждая живая подписка платит примерно один раз;
- **тариф**: цену меняли по тарифам (199 -> 249 и 399 -> 449), и эффект цены видно только внутри тарифа;
- **канал**: чтобы ответить "где" (аддитивная модель, Л6): не ушли ли после повышения клиенты одного канала.

Мельче (платёж или подписка) не нужно: вопрос про выручку компании, а не про отдельного клиента. Крупнее (просто месяц) нельзя:
пропадут ответы "где" и эффект структуры тарифов.

**Как считаем каждую колонку:**

| Колонка | Определение |
|---|---|
| активные подписки (`active`) | подписки, активные в месяце: начались до конца месяца и не закончились до его начала (как в С6) |
| число платежей (`n_payments`) | успешные платежи без двойных списаний (таблица `paid`) |
| выручка (`revenue`) | сумма этих платежей, руб. |
| дней с данными (`days`) | сколько дней месяца есть в данных: у апреля 28 из 30 из-за дыры 14-15.04 |

In [ ]:
MONTHS = pd.period_range("2025-12", "2026-06", freq="M")   # 7 месяцев: три до повышения, март, три после

# Канал у подписки это канал её владельца. Пропуск заменяем на "без канала":
# иначе groupby и pivot_table молча выбросят эти строки (Л3, слайд 25; С6, раздел 2).
subs["channel"] = subs["acquisition_channel"].fillna("без канала")

# Приписываем каждому платежу канал его подписки.
# Кратность m:1: у подписки много платежей, у платежа ровно одна подписка. Если это не так, merge упадёт.
rows_before = len(paid)
paid = paid.merge(subs[["subscription_id", "channel"]], on="subscription_id", how="left", validate="m:1")
assert len(paid) == rows_before, "соединение размножило платежи"
assert paid["channel"].notna().all(), "у части платежей не нашлась подписка"
print("платежей:", len(paid), "| каналы:", sorted(paid["channel"].unique()))

**Вывод.** Каждому из 1 210 518 платежей приписан канал владельца, соединение с проверкой кратности (`validate="m:1"`)
не упало и не размножило строки. Каналов 7: шесть настоящих и "без канала".

**Следующий шаг: посчитать активные подписки, платежи и выручку в зерне месяц × тариф × канал** и сверить деньги
с рабочей таблицей: если сборка потеряет или добавит рубли, проверка упадёт.

In [ ]:
# Активные подписки по месяцам. Цикл по 7 месяцам, внутри каждого векторная маска: это быстро.
parts = []
for m in MONTHS:
    start = m.start_time.tz_localize("UTC")          # 1-е число месяца, 00:00 UTC
    end = (m + 1).start_time.tz_localize("UTC")      # 1-е число следующего месяца: правый конец не включён
    # Активна в месяце: началась раньше конца месяца И (не закончилась ИЛИ закончилась не раньше его начала).
    alive = subs[(subs["started_at"] < end) & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    g = alive.groupby(["tariff_id", "channel"]).size().rename("active").reset_index()
    g["month"] = m
    parts.append(g)
active = pd.concat(parts, ignore_index=True)

# Платежи и выручка в том же зерне: месяц × тариф × канал.
rev = (paid.groupby(["month", "tariff_id", "channel"], as_index=False)
           .agg(n_payments=("payment_id", "size"), revenue=("amount", "sum")))

# Соединяем 1:1: в обеих таблицах одна строка на месяц × тариф × канал.
# outer + indicator показывают, есть ли платежи у клеток без активных подписок, и наоборот.
monthly = active.merge(rev, on=["month", "tariff_id", "channel"], how="outer", validate="1:1", indicator=True)
print(monthly["_merge"].value_counts())
monthly = monthly.drop(columns="_merge").fillna({"active": 0, "n_payments": 0, "revenue": 0})

# ДЕНЬГИ: выручка в витрине обязана совпасть с суммой paid до копейки.
assert abs(monthly["revenue"].sum() - paid["amount"].sum()) < 0.01, "деньги потерялись при сборке"
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
print("строк в monthly:", len(monthly), "(7 месяцев × 2 тарифа × 7 каналов = 98)")
monthly.head()

**Вывод.** В таблице (`monthly`) 98 строк: 7 месяцев × 2 тарифа × 7 каналов. Все 98 клеток нашли пару
(`both`): нет клеток с платежами, но без активных подписок, и наоборот. Выручка сошлась с рабочей таблицей до копейки.

**Следующий шаг: свести таблицу по месяцам и разложить выручку на три множителя в пересчёте на день.**
Месяцы разной длины (февраль 28 дней) и апрель с дырой иначе сравнивать нельзя (Л5, слайд 18).

In [ ]:
# Сводка по месяцам. Сколько дней каждого месяца есть в данных: у апреля дыра в 2 дня.
paid["day"] = paid["paid_at"].dt.floor("D")
days_with_data = paid.groupby("month")["day"].nunique()

m = monthly.groupby("month")[["active", "n_payments", "revenue"]].sum()
m["days"] = days_with_data
m["days_in_month"] = m.index.days_in_month               # сколько дней в календарном месяце (Л5, слайд 18)

# Средняя выручка в день = активные подписки × среднее число платежей на подписку в день × средняя цена платежа.
# В пересчёте на день месяцы разной длины становятся сравнимы, а апрель считается только по дням с данными.
m["revenue_per_day"] = m["revenue"] / m["days"]                # средняя выручка в день, руб.
m["per_sub_day"] = m["n_payments"] / m["active"] / m["days"]   # среднее число платежей на одну активную подписку в день
m["price"] = m["revenue"] / m["n_payments"]                    # средняя цена одного платежа, руб.

# Проверка, что три множителя дают среднюю выручку в день точно: модель обязана сходиться.
assert np.allclose(m["active"] * m["per_sub_day"] * m["price"], m["revenue_per_day"])

m["growth_per_day_pct"] = m["revenue_per_day"].pct_change() * 100   # прирост средней выручки в день к прошлому месяцу, %

# Для показа: среднее число платежей на 1000 активных подписок в день. Это то же per_sub_day, умноженное на 1000:
# иначе при двух знаках после запятой все месяцы выглядят одинаково (0.03).
# Показываем показатели в строках, месяцы в столбцах слева направо.
m.assign(per_1000_subs_day=m["per_sub_day"] * 1000).drop(columns="per_sub_day").T

**Вывод.** Помесячно (сравнение окон будет в разделе 3):

| Показатель | Декабрь | Январь | Февраль | Март | Апрель | Май | Июнь |
|---|---:|---:|---:|---:|---:|---:|---:|
| средняя выручка в день (`revenue_per_day`), тыс. руб. | 1 469 | 1 512 | 1 552 | 1 876 | 1 919 | 1 958 | 2 003 |
| активные подписки (`active`), тыс. | 170,2 | 175,1 | 178,7 | 183,8 | 187,6 | 191,7 | 195,5 |
| среднее число платежей на 1000 подписок в день (`per_1000_subs_day`) | 31,42 | 31,41 | 31,60 | 31,44 | 31,48 | 31,44 | 31,52 |
| средняя цена платежа (`price`), руб. | 274,76 | 274,85 | 274,85 | 324,84 | 324,97 | 324,90 | 324,98 |

- Средняя выручка в день скачком выросла в марте (+20,9% к февралю) и дальше держится на новом уровне.
- Средняя цена платежа сменилась ровно один раз, в марте, и все следующие месяцы стоит на 325 руб.
- База активных подписок растёт каждый месяц, и до повышения, и после: обвала нет.
- Среднее число платежей на подписку в день почти постоянно. Февраль выделяется из ряда: похоже, это особенность
  короткого месяца в определении "активна в месяце", а не реакция клиентов.
- После марта средняя выручка в день растёт на 2,1-2,3% в месяц, до повышения росла на 2,7-2,9%. Это надо проверить:
  либо рост замедлился, либо просто выросла база для процента.

**Следующий шаг: проверить структуру тарифов.** Средняя цена платежа может расти и без изменения цен, если клиенты
перешли на дорогой тариф (Л6, слайд 15). Тогда рост цены в таблице выше был бы не только эффектом повышения.

In [ ]:
# То же по тарифам: средняя цена платежа внутри тарифа и доля тарифа в платежах.
# Средняя цена смешивает цену и структуру (Л6, слайд 15): если доля дорогого тарифа выросла,
# средний чек растёт и без изменения цен. Поэтому смотрим долю отдельно.
t = monthly.pivot_table(index="month", columns="tariff", values=["n_payments", "revenue", "active"], aggfunc="sum")
by_tariff = pd.DataFrame({
    "price_basic": t["revenue"]["Базовый"] / t["n_payments"]["Базовый"],
    "price_ext": t["revenue"]["Расширенный"] / t["n_payments"]["Расширенный"],
    "share_ext_pct": t["n_payments"]["Расширенный"] / t["n_payments"].sum(axis=1) * 100,   # доля "Расширенного" в платежах, %
    "active_basic": t["active"]["Базовый"],
    "active_ext": t["active"]["Расширенный"],
})
by_tariff.T   # показатели в строках, месяцы в столбцах

**Вывод.** Средняя цена платежа внутри тарифа ровно равна цене из справочника: 199 -> 249 руб. у "Базового" (`price_basic`)
и 399 -> 449 руб. у "Расширенного" (`price_ext`). Доля "Расширенного" в платежах (`share_ext_pct`) держится на 37,9-38,0%
все 7 месяцев, а активные подписки обоих тарифов (`active_basic`, `active_ext`) растут. Массового перехода на дешёвый тариф нет,
значит эффект структуры почти нулевой и рост средней цены платежа это действительно повышение цен.

**Следующий шаг: факторное разложение.** Теперь известны все три множителя, и можно точно сказать,
сколько рублей в день дал каждый из них при сравнении 3 месяцев до и 3 месяцев после, со сверкой суммы влияний (`assert`).

## 3. Факторное разложение выручки

Сравниваем **два окна по 3 месяца**: до повышения (декабрь 2025 - февраль 2026) и после (март - май 2026).
Один месяц может быть случайным, а за 3 месяца каждая подписка успевает продлиться по новой цене трижды.
Июнь не входит в окно "после": по нему проверим, что эффект держится (раздел 3.5).

Показатель: **средняя выручка в день за окно** = выручка окна / дни окна с данными. В днях, потому что окна разной длины
(в декабре-феврале 90 дней, в марте-мае 92, из них с данными 90 из-за дыры 14-15 апреля).

Модель: средняя выручка в день = среднее число активных подписок × среднее число платежей на подписку в день × средняя цена платежа.

| Множитель | Как считаем за окно |
|---|---|
| среднее число активных подписок (`active`) | подписко-дни окна / дни окна: месяц весит столько, сколько в нём дней с данными |
| среднее число платежей на подписку в день (`per_sub_day`) | платежи окна / подписко-дни окна |
| средняя цена платежа (`price`) | выручка окна / платежи окна |

Перемножение трёх множителей даёт ровно выручку окна / дни окна. Порядок как на Л6: сначала "где" (сложение по частям),
потом "за счёт чего" (умножение), потом сочетание моделей. Сумма влияний везде проверяется `assert`.

### 3.1 Где: тарифы и каналы

In [ ]:
BEFORE = pd.period_range("2025-12", "2026-02", freq="M")   # окно "до": 3 месяца по старым ценам
AFTER = pd.period_range("2026-03", "2026-05", freq="M")    # окно "после": 3 месяца по новым ценам
W0, W1 = "дек 2025 - фев 2026", "мар - май 2026"          # подписи окон, идут слева направо по времени

monthly["days"] = monthly["month"].map(days_with_data)          # дни месяца с данными
monthly["active_days"] = monthly["active"] * monthly["days"]    # подписко-дни: активные подписки × дни
win = monthly[monthly["month"].isin(BEFORE.union(AFTER))].copy()
win["window"] = np.where(win["month"].isin(BEFORE), W0, W1)


def window_days(df: pd.DataFrame) -> int:
    """Сколько дней с данными в окне: у каждого месяца одно число дней, поэтому берём по одной строке на месяц."""
    return df.drop_duplicates("month")["days"].sum()


def window_factors(df: pd.DataFrame) -> dict:
    """Три множителя средней выручки в день за окно (строки df это клетки месяц × тариф × канал одного окна)."""
    days, active_days = window_days(df), df["active_days"].sum()
    return {
        "active": active_days / days,                         # среднее число активных подписок
        "per_sub_day": df["n_payments"].sum() / active_days,  # среднее число платежей на подписку в день
        "price": df["revenue"].sum() / df["n_payments"].sum(),  # средняя цена платежа
    }


DAYS = {w: window_days(win[win["window"] == w]) for w in (W0, W1)}
x0 = window_factors(win[win["window"] == W0])
x1 = window_factors(win[win["window"] == W1])
rpd0 = win.loc[win["window"] == W0, "revenue"].sum() / DAYS[W0]   # средняя выручка в день, окно "до"
rpd1 = win.loc[win["window"] == W1, "revenue"].sum() / DAYS[W1]   # средняя выручка в день, окно "после"
total_delta = rpd1 - rpd0

# Проверка модели: произведение трёх множителей обязано дать среднюю выручку в день.
assert abs(math.prod(x0.values()) - rpd0) < 0.01 and abs(math.prod(x1.values()) - rpd1) < 0.01


def where(by: str) -> pd.DataFrame:
    """Аддитивная модель: средняя выручка в день по частям (тариф или канал) в двух окнах."""
    t = win.pivot_table(index=by, columns="window", values="revenue", aggfunc="sum")[[W0, W1]]  # окна слева направо
    t = t.div(pd.Series(DAYS))                                 # выручка окна / дни окна = средняя выручка в день
    t["изменение"] = t[W1] - t[W0]                             # влияние части на итог, руб. в день
    t["изменение_pct"] = t["изменение"] / t[W0] * 100          # рост части, %
    t["доля_роста_pct"] = t["изменение"] / total_delta * 100   # какую долю итогового роста дала часть, %
    # Сумма частей равна целому: иначе мы что-то потеряли (как 11 357 руб. без канала в С6).
    assert abs(t["изменение"].sum() - total_delta) < 0.01, "части не сходятся с итогом"
    return t.sort_values("изменение", ascending=False)


print("дней с данными:", DAYS)
print(f"средняя выручка в день: {rpd0:,.0f} -> {rpd1:,.0f} руб., изменение {total_delta:,.0f} руб. ({total_delta / rpd0:.1%})".replace(",", " "))
where("tariff")

**Вывод.** В обоих окнах по 90 дней с данными. Средняя выручка в день выросла с 1 509 487 до 1 917 864 руб.: **+408 377 руб. в день, +27,1%**.
"Базовый" дал +233 371 руб. в день (+34,4%, 57% роста), "Расширенный" +175 005 руб. в день (+21,1%, 43% роста).
Сумма частей сходится с итогом (`assert`). Рост каждого тарифа больше роста его цены (+25,1% и +12,5%):
сверху добавился рост базы подписок, который шёл и до повышения.

**Следующий шаг: та же разбивка по каналам.** Так проверяем гипотезу 3: не просели ли после повышения клиенты какого-то одного канала.

In [ ]:
where("channel")

**Вывод.** Все 7 каналов выросли: от +23,4% (`social`) до +28,8% (`organic`). Ни один канал не упал,
и доля каждого канала в росте (`доля_роста_pct`) близка к его доле в выручке.
Меньше всех вырос `social`: у него самый высокий отток (раздел 4.2), поэтому его база растёт медленнее.
По выручке гипотеза 3 в части каналов **не подтверждается**. Отток по каналам проверяем отдельно в разделе 4.2.

**Следующий шаг: мультипликативная модель.** Аддитивная ответила "где": везде. Теперь нужно понять,
"за счёт чего": база, частота платежей или цена.

### 3.2 За счёт чего: три множителя

In [ ]:
FACTORS = ["active", "per_sub_day", "price"]     # порядок курса: сначала "сколько", потом "почём"
LABELS = {"active": "среднее число активных подписок", "per_sub_day": "среднее число платежей на подписку в день",
          "price": "средняя цена платежа"}

# Цепные подстановки: меняем факторы по одному в порядке FACTORS (функция contrib из src/prime/factors.py).
effects = pd.Series(contrib(x0, x1, FACTORS))
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению средней выручки в день"

out = pd.DataFrame({
    W0: pd.Series(x0),
    W1: pd.Series(x1),
    "изменение_pct": (pd.Series(x1) / pd.Series(x0) - 1) * 100,
    "влияние_руб_в_день": effects,
    "доля_роста_pct": effects / total_delta * 100,
}).rename(index=LABELS)
# Для показа: платежи на подписку в день умножаем на 1000, иначе при двух знаках все значения выглядят как 0.03.
out.loc[LABELS["per_sub_day"], [W0, W1]] *= 1000
out.rename(index={LABELS["per_sub_day"]: "среднее число платежей на 1000 подписок в день"})

**Вывод.** Из +408 377 руб. в день:

- средняя цена платежа (`price`) дала **+295 615 руб. в день, 72,4% роста**: 274,82 -> 324,90 руб. (+18,2%);
- среднее число активных подписок (`active`) дало +113 697 руб. в день, 27,8%: 174 526 -> 187 672 (+7,5%);
- среднее число платежей на подписку в день (`per_sub_day`) почти не изменилось: -935 руб. в день, -0,2%.

Сумма трёх влияний равна изменению итога (`assert`). Повышение цен приносит около 296 тыс. руб. в день, примерно 8,9 млн руб. за 30 дней.
Влияние базы больше, чем на одном переходе февраль -> март (раздел 3.5), потому что между серединами окон 3 месяца обычного роста базы.
Гипотеза 1 **подтверждается**: почти три четверти роста дала цена, а остальное дал рост базы, который шёл и до повышения.

**Следующий шаг: проверить, не зависит ли вывод от порядка подстановок.** Совместный член достаётся фактору,
который меняют позже (Л6, слайды 10-11), а здесь выросли сразу два фактора: и база, и цена.

In [ ]:
import itertools

# Порядок подстановок меняет ответ: совместный член достаётся фактору, который меняют позже (Л6, слайды 10-11).
# Проверяем все 6 порядков. Если вывод одинаковый во всех, он не зависит от нашего выбора порядка.
orders = pd.DataFrame({" -> ".join(o): contrib(x0, x1, list(o)) for o in itertools.permutations(FACTORS)}).T
orders = orders[FACTORS].rename(columns=LABELS)
orders.loc["среднее по 6 порядкам"] = orders.mean()     # среднее по порядкам от порядка не зависит (вектор Шепли, С6)
orders.round(0)

**Вывод.** Во всех 6 порядках вывод один и тот же. Влияние средней цены платежа от 274 908 до 295 786 руб. в день,
среднего числа активных подписок от 113 632 до 134 416, среднего числа платежей на подписку от -1 106 до -870.
Разброс около 21 тыс. руб. в день: это совместный член роста базы (+7,5%) и цены (+18,2%). При любом порядке цена даёт
больше чем вдвое против базы. Среднее по 6 порядкам: цена +285 343, база +124 020, платежи на подписку -986 руб. в день.
В записке используем порядок курса: сначала количество, потом цена.

**Следующий шаг: отделить цену от структуры тарифов.** Средняя цена платежа могла вырасти и потому,
что клиенты перешли на дорогой тариф (Л6, слайд 15).

### 3.3 Цена или структура тарифов

In [ ]:
# Средняя цена платежа = сумма по тарифам (доля тарифа в платежах × средняя цена платежа тарифа).
# Её изменение раскладываем на две части (Л6, слайд 15):
#   структура: доли тарифов поменялись, цены ещё старые;
#   цена: цены поменялись при новых долях.
tp = win.pivot_table(index="tariff", columns="window", values=["n_payments", "revenue"], aggfunc="sum")
share0 = tp["n_payments"][W0] / tp["n_payments"][W0].sum()   # доли тарифов в платежах, окно "до"
share1 = tp["n_payments"][W1] / tp["n_payments"][W1].sum()   # доли тарифов в платежах, окно "после"
p0 = tp["revenue"][W0] / tp["n_payments"][W0]                # средняя цена платежа тарифа, окно "до"
p1 = tp["revenue"][W1] / tp["n_payments"][W1]                # средняя цена платежа тарифа, окно "после"

structure = ((share1 - share0) * p0).sum()     # руб. на платёж за счёт смены долей
pure_price = (share1 * (p1 - p0)).sum()        # руб. на платёж за счёт самих цен
assert abs(structure + pure_price - (x1["price"] - x0["price"])) < 1e-9, "цена и структура не сходятся"

# В рубли в день: в contrib цена стоит последней, её влияние = активные(после) × платежи на подписку(после) × изменение цены.
scale = x1["active"] * x1["per_sub_day"]       # среднее число платежей в день в окне "после"
split = pd.Series({"структура тарифов": structure * scale, "цены тарифов": pure_price * scale})
assert abs(split.sum() - effects["price"]) < 0.01, "не сходится с влиянием цены из 3.2"

print(pd.DataFrame({f"доля, {W0}, %": share0 * 100, f"доля, {W1}, %": share1 * 100,
                    f"средняя цена, {W0}": p0, f"средняя цена, {W1}": p1}).round(3))
print()
print("влияние, руб. в день:")
print(split.round(0))

**Вывод.** Доля "Расширенного" в платежах почти не изменилась: 37,91% до и 37,95% после.
Структура тарифов дала +468 руб. в день, сами цены тарифов +295 147 руб. в день. Вместе это ровно влияние
средней цены платежа из 3.2 (`assert`). Клиенты не стали переходить на дешёвый "Базовый":
гипотеза 3 в части тарифов **не подтверждается**.

**Следующий шаг: сочетание моделей**, канал × фактор. Проверяем, что картина "цена дала основное" держится в каждом канале,
а не только в среднем.

### 3.4 Сочетание моделей: канал × фактор

In [ ]:
# Внутри каждого канала раскладываем его изменение на те же три множителя (Л6, слайд 14).
rows = {}
for c, g in win.groupby("channel"):
    rows[c] = contrib(window_factors(g[g["window"] == W0]), window_factors(g[g["window"] == W1]), FACTORS)
by_ch_factor = pd.DataFrame(rows).T[FACTORS]
by_ch_factor["всего"] = by_ch_factor.sum(axis=1)
by_ch_factor = by_ch_factor.sort_values("всего", ascending=False)
by_ch_factor.loc["итого"] = by_ch_factor.sum()

# Сумма всех клеток равна изменению итога.
assert abs(by_ch_factor.loc["итого", "всего"] - total_delta) < 0.01, "каналы × факторы не сходятся с итогом"
by_ch_factor.rename(columns=LABELS).round(0)

**Вывод.** В каждом канале главный фактор это средняя цена платежа: от +5 983 руб. в день ("без канала") до +83 795 (`organic`).
Среднее число активных подписок выросло во всех каналах: от +2 600 до +38 239 руб. в день. Среднее число платежей на подписку
почти не изменилось нигде (от -462 до +72 руб. в день). Сумма всех клеток равна изменению итога, 408 377 руб. в день (`assert`).
Итоги по факторам чуть отличаются от 3.2 (например, 113 764 против 113 697 у базы): внутри каналов свои средние, а сумма всего та же.

**Следующий шаг: контроль на всех переходах месяц к месяцу.** Если эффект цены настоящий, он должен появиться один раз,
в марте, а на переходах без смены цен влияние средней цены платежа должно быть около нуля.

### 3.5 Контроль: все переходы месяц к месяцу

In [ ]:
# Если эффект цены настоящий, он должен появиться один раз, на переходе февраль -> март,
# а на остальных переходах (цены не менялись) влияние средней цены платежа должно быть около нуля.
# Июнь здесь тоже есть: по нему видно, держится ли эффект после окна "после".
steps = {}
for a, b in zip(m.index[:-1], m.index[1:]):
    eff = contrib(m.loc[a, FACTORS].to_dict(), m.loc[b, FACTORS].to_dict(), FACTORS)
    assert abs(sum(eff.values()) - (m.loc[b, "revenue_per_day"] - m.loc[a, "revenue_per_day"])) < 0.01
    steps[f"{a} -> {b}"] = eff
all_steps = pd.DataFrame(steps)                 # факторы в строках, переходы в столбцах слева направо
all_steps.loc["всего"] = all_steps.sum()
all_steps.rename(index=LABELS).round(0)

**Вывод.**

- Влияние средней цены платежа появляется один раз: +288 723 руб. в день на переходе февраль -> март.
  На остальных пяти переходах от -422 до +781 руб. в день, почти ноль. Значит, рост это эффект повышения, а не сезонность.
- **Эффект держится и после окна "после":** на переходе май -> июнь влияние цены +530 руб. в день, то есть цена не откатилась,
  а средняя цена платежа в июне та же, 324,98 руб. (раздел 2).
- База каждый месяц добавляет примерно одинаково, от 31 до 44 тыс. руб. в день, и до повышения, и после.
- **В рублях рост после повышения не замедлился:** +42 413, +39 591, +44 736 руб. в день за апрель-июнь против +42 994 и +40 179 до марта.
  Процент роста стал меньше (раздел 2) только потому, что сама выручка стала на 21% больше. Вопрос из раздела 2 закрыт.
- Среднее число платежей на подписку прыгает от -8 302 до +9 515 руб. в день без связи с ценой: это длина месяцев
  и определение "активна в месяце". В окнах по 3 месяца эти колебания гасят друг друга (-935 руб. в день в 3.2).

**Следующий шаг: отток (раздел 4).** Рост базы в рублях держится, но он может скрывать одновременный рост и новых подписок, и оттока.
Гипотезу 2 проверяем напрямую.

## 4. Проверка гипотезы про отток

**Гипотеза 2:** после повышения цен клиенты стали чаще уходить.

Определение оттока как на Л5 и С5: **доля подписок, активных на начало месяца, которые закончились до его конца**.
Знаменатель фиксируется до начала месяца: подписки, оформленные внутри месяца, в него не входят. Иначе хороший месяц
продаж сам по себе "снижал" бы отток. Считаем функцией оттока (`churn_rate`) из `src/prime/metrics.py`.

**Средний дневной отток.** Месяцы разной длины: в феврале 28 дней, и отток за февраль меньше просто потому, что дней меньше.
Поэтому переводим отток месяца в средний дневной. Доли не делят на дни, а перемножают, как годовой отток на Л5:
средний дневной отток = `1 - (1 - отток месяца) ** (1 / дней)` (функция `daily_churn`). Для чтения показываем его
в пересчёте на 30 дней: `1 - (1 - дневной) ** 30` (функция `churn_30d`).

**Окна сравнения те же, что в разделе 3:** декабрь-февраль против марта-мая. Ряд показываем шире, с сентября 2025 по июнь 2026,
чтобы было видно обычные колебания. Подписки в базе полные (дыра 14-15 апреля только в платежах), поэтому апрель здесь обычный месяц.

**Слепое пятно определения:** подписки, оформленные и отменённые в одном месяце, не попадают ни в один месяц оттока.
А это как раз те, кто первым увидел новую цену при оформлении. Их проверяем отдельно в разделе 4.6.

### 4.1 Отток и приток по месяцам

In [ ]:
CHURN_MONTHS = pd.period_range("2025-09", "2026-06", freq="M")

rows = {}
for p in CHURN_MONTHS:
    start = p.start_time.tz_localize("UTC")
    end = (p + 1).start_time.tz_localize("UTC")
    alive = subs[active_on(subs, start)]                  # активные на 1-е число месяца (знаменатель оттока)
    rows[p] = {
        "active_start": len(alive),                                                       # активные на начало месяца
        "churned": int((alive["ended_at"] < end).sum()),                                  # из них закончились в месяце
        "new": int(((subs["started_at"] >= start) & (subs["started_at"] < end)).sum()),   # новые подписки за месяц
        "days": p.days_in_month,
    }
flow = pd.DataFrame(rows)                    # показатели в строках, месяцы в столбцах слева направо

churn_m = flow.loc["churned"] / flow.loc["active_start"]                          # отток месяца, доля
daily = pd.Series({p: daily_churn(churn_m[p], flow.loc["days", p]) for p in flow.columns})  # средний дневной отток
flow.loc["churn_month_pct"] = churn_m * 100
flow.loc["churn_daily_pct"] = daily * 100
flow.loc["churn_30d_pct"] = daily.map(churn_30d) * 100                             # отток в пересчёте на 30 дней
flow.loc["new_per_day"] = flow.loc["new"] / flow.loc["days"]                       # среднее число новых подписок в день

# Сверка с функцией из src/prime/metrics.py: два разных кода обязаны дать одно число.
assert abs(churn_rate(subs, "2026-03-01", "2026-04-01") - churn_m[pd.Period("2026-03", "M")]) < 1e-12

b, a = flow[BEFORE], flow[AFTER]
print(f"средний отток за 30 дней: {W0} {b.loc['churn_30d_pct'].mean():.2f}%, {W1} {a.loc['churn_30d_pct'].mean():.2f}%")
print(f"средний дневной отток:    {W0} {b.loc['churn_daily_pct'].mean():.4f}%, {W1} {a.loc['churn_daily_pct'].mean():.4f}%")
print(f"среднее число новых подписок в день: {W0} {b.loc['new_per_day'].mean():.0f}, {W1} {a.loc['new_per_day'].mean():.0f}")
flow.loc[["active_start", "churned", "churn_month_pct", "churn_30d_pct", "new", "new_per_day"]]

**Вывод.** Отток не вырос. Средний отток в пересчёте на 30 дней (`churn_30d_pct`): **3,91% в декабре-феврале и 3,88% в марте-мае**.
Средний дневной отток (`churn_daily_pct`) 0,1329% и 0,1317%. Помесячно отток от 3,85% до 3,97% все 10 месяцев, март (3,89%) ничем не выделяется.
Приток в пределах обычных колебаний: среднее число новых подписок в день (`new_per_day`) 366 до и 360 после, а за все 10 месяцев от 354 до 369.
Без пересчёта на день февраль с оттоком 3,71% (`churn_month_pct`) выглядел бы лучшим месяцем, хотя в нём просто 28 дней.
Сверка с функцией оттока (`churn_rate`) прошла (`assert`).

**Следующий шаг: отток по тарифам и каналам.** Средний отток по всей базе мог не измениться, если в одной группе он вырос,
а в другой упал.

### 4.2 Отток по тарифам и каналам: до и после

In [ ]:
def churn_table(by: str) -> pd.DataFrame:
    """Отток в пересчёте на 30 дней, %: группы в строках, месяцы в столбцах, плюс средние по окнам."""
    res = {}
    for p in CHURN_MONTHS:
        start, end = p.start_time.strftime("%Y-%m-%d"), (p + 1).start_time.strftime("%Y-%m-%d")
        # groupby + apply: функция оттока получает подписки одной группы (как в С5).
        month_churn = subs.groupby(by).apply(churn_rate, start, end)
        res[p] = month_churn.map(lambda c: churn_30d(daily_churn(c, p.days_in_month))) * 100
    t = pd.DataFrame(res)
    t[f"среднее {W0}"] = t[BEFORE].mean(axis=1)
    t[f"среднее {W1}"] = t[AFTER].mean(axis=1)
    t["разница_пп"] = t[f"среднее {W1}"] - t[f"среднее {W0}"]     # в процентных пунктах
    return t


subs["tariff"] = subs["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
churn_by_tariff = churn_table("tariff")
churn_by_tariff.round(2)

**Вывод.** Отток не вырос ни в одном тарифе. "Базовый": средний отток 3,95% до и 3,86% после (-0,09 п.п.),
хотя его цена выросла сильнее всего, на 25,1%. "Расширенный": 3,85% и 3,91% (+0,05 п.п.).
Обе разницы меньше обычных колебаний от месяца к месяцу (от 3,82% до 4,02%), то есть это шум.

**Следующий шаг: каналы.** Клиенты разных каналов по-разному держатся за подписку (отток `social` втрое выше, чем у `organic`),
и реакция на цену могла спрятаться в одном канале.

In [ ]:
churn_by_channel = churn_table("channel")
show_cols = list(BEFORE) + list(AFTER) + [f"среднее {W0}", f"среднее {W1}", "разница_пп"]
churn_by_channel[show_cols].sort_values("разница_пп").round(2)

**Вывод.** У пяти каналов из семи отток после повышения такой же или ниже: разница от -0,15 до -0,03 п.п.
Выросло у двух каналов с самым высоким оттоком:

- `social`: 6,89% -> 7,12% (+0,23 п.п.). Все три месяца после (7,23%, 7,13%, 7,00%) выше всех трёх месяцев до (от 6,83% до 6,97%);
- `performance`: 5,26% -> 5,38% (+0,12 п.п.). Тоже все три месяца после выше месяцев до, но разница меньше.

Это **слабый сигнал, на грани шума**: разница в десятые доли процентного пункта, на всю выручку он почти не влияет.
Но он логичен: оба канала платные (реклама в соцсетях и performance-маркетинг), и такие клиенты, скорее всего, чувствительнее к цене. Гипотеза 3 по оттоку
**частично подтверждается** для `social` и `performance`, это пойдёт в записку в раздел "что делать".

**Следующий шаг: отмены по дням вокруг 1 марта.** Средние за месяц могли сгладить короткий всплеск сразу после первого
списания по новой цене (Л6, слайд 25: сглаживание прячет событие).

### 4.3 Отмены по дням вокруг 1 марта

In [ ]:
# Если клиенты уходят из-за цены, всплеск отмен должен быть сразу после первого списания по новой цене,
# то есть в первые дни и недели марта. Смотрим отмены по дням с 1 февраля по 31 марта.
ended = subs.loc[subs["ended_at"].notna(), "ended_at"].dt.tz_localize(None).dt.floor("D")
daily_end = ended.value_counts().sort_index().loc["2026-02-01":"2026-03-31"]
daily_end = daily_end.reindex(pd.date_range("2026-02-01", "2026-03-31", freq="D"), fill_value=0)   # дни без отмен тоже показываем

print(f"среднее число отмен в день, февраль: {daily_end['2026-02'].mean():.0f}")
print(f"среднее число отмен в день, 1-14 марта: {daily_end['2026-03-01':'2026-03-14'].mean():.0f}")
print(f"среднее число отмен в день, 15-31 марта: {daily_end['2026-03-15':'2026-03-31'].mean():.0f}")
print(f"больше всего отмен в день: {daily_end.max()} ({daily_end.idxmax().date()})")
print()
# По неделям: так виден тренд без шума отдельных дней. Недели в столбцах слева направо.
weekly = daily_end.resample("W-SUN").sum()
weekly.index = weekly.index.strftime("неделя до %d.%m")
weekly.to_frame("отмен за неделю").T

**Вывод.** Всплеска нет. Среднее число отмен в день: 223 в феврале, 222 за 1-14 марта, 226 за 15-31 марта.
Самый большой день (254 отмены, 30 марта) выше среднего меньше чем на 15% и стоит в конце месяца, а не сразу после повышения.
По неделям от 1 510 до 1 601 отмены. Первый и последний столбцы (214 и 498) это неполные недели: только 1 февраля и только 30-31 марта.

**Следующий шаг: причины отмен.** Даже если отмен столько же, внутри них могла вырасти доля тех, кто ушёл из-за цены.

### 4.4 Причины отмен

In [ ]:
# Причина отмены лежит в подписках. Берём отдельным запросом и соединяем 1:1 по подписке.
reasons = q("select subscription_id, cancel_reason from prime.subscriptions where ended_at is not null")
ends = subs[subs["ended_at"].notna()].merge(reasons, on="subscription_id", how="left", validate="1:1")
ends["end_month"] = ends["ended_at"].dt.tz_localize(None).dt.to_period("M")
ends = ends[ends["end_month"].isin(CHURN_MONTHS)]

# Доля каждой причины среди отмен месяца, %. Пустую причину заменяем на "не указана", чтобы она не пропала.
# Причины в строках, месяцы в столбцах слева направо.
reason_share = pd.crosstab(ends["cancel_reason"].fillna("не указана"), ends["end_month"], normalize="columns") * 100
print("всего отмен:", len(ends))
reason_share.round(1)

**Вывод.** Причины отмен не изменились. Доля причины "слишком дорого" (`too_expensive`) от 23,3% до 25,1% все 10 месяцев:
24,3% в феврале и 24,1% в марте. Самое большое значение в июне, 25,1%, но оно лишь на 0,2 п.п. выше января (24,9%), до повышения.
Доля "не проходит платёж" (`payment_failed`) от 12,9% до 14,6%, без скачка в марте.
Остальные причины ("не пользуюсь" `not_using`, "перешёл" `switched`, "другое" `other`) тоже на месте.
Пустых причин нет: у всех 65 198 отмен она указана.

**Следующий шаг: неудачные платежи.** Новая цена могла не проходить по картам и не приводить к отмене сразу,
а копиться в неудачных списаниях.

### 4.5 Неудачные платежи

In [ ]:
# Если новая цена не проходит по картам, вырастет доля неудачных списаний (failed).
st = pay.assign(month=pay["paid_at"].dt.tz_localize(None).dt.to_period("M"))
# Статусы в строках, месяцы в столбцах: доля статуса среди всех попыток списания месяца, %.
status_share = pd.crosstab(st["status"], st["month"], normalize="columns") * 100
status_share.round(2)

**Вывод.** Доля неудачных платежей (`failed`) от 3,24% до 3,38% во все месяцы: 3,29% в феврале и 3,32% в марте.
Доля возвратов (`refunded`) около 1,2%. Новая цена по картам проходит так же, как старая.

**Следующий шаг: ранний отток новых подписок.** Осталось слепое пятно определения оттока: те, кто оформил подписку
уже по новой цене и сразу ушёл.

### 4.6 Ранний отток новых подписок

Закрываем слепое пятно из начала раздела: те, кто оформил подписку уже по новой цене и быстро ушёл.

Сначала смотрим, сколько вообще живут закончившиеся подписки: от этого зависит, какое окно считать "ранним".
Затем для подписок, оформленных в каждом месяце, считаем долю тех, кто **ушёл на первом продлении**, то есть не продлил подписку ни разу.
Окно одинаковое для всех месяцев оформления (Л5, слайд 23: у воронки есть окно), поэтому сравнение честное.
Триалы показываем отдельно: пробная подписка может заканчиваться быстро по своей природе.

In [ ]:
new_subs = subs.assign(start_month=subs["started_at"].dt.tz_localize(None).dt.to_period("M"))
new_subs = new_subs[new_subs["start_month"].isin(CHURN_MONTHS)]
# Сколько живут закончившиеся подписки: от этого зависит, какое окно "раннего" ухода имеет смысл.
lifetime = (new_subs["ended_at"] - new_subs["started_at"]).dt.total_seconds() / 86400   # дней жизни
print("дней жизни у закончившихся подписок:")
print(lifetime.describe(percentiles=[0.01, 0.05, 0.25, 0.5]).round(1).to_string())
print("закончились до 30 дней:", int((lifetime < 30).sum()), "| ровно на 30-31 день:", int(((lifetime >= 30) & (lifetime < 32)).sum()))
print()
# Ушла на первом продлении: прожила меньше 32 дней (первое продление через 30 дней, плюс запас на часы и минуты).
# Последний месяц оформления, июнь 2026: 32 дня от 30 июня кончаются 2 августа, это внутри данных (до 30 августа).
new_subs["early_end"] = lifetime < 32
new_subs["kind"] = np.where(new_subs["is_trial"], "триал", "без триала")

# Разрезы в строках (все, тариф, триал), месяцы оформления в столбцах слева направо. В клетке: доля ранних отмен, %.
early = pd.concat([
    new_subs.groupby("start_month")["early_end"].mean().to_frame("все новые подписки").T,
    new_subs.pivot_table(index="tariff", columns="start_month", values="early_end", aggfunc="mean"),
    new_subs.pivot_table(index="kind", columns="start_month", values="early_end", aggfunc="mean"),
]) * 100
early[f"среднее {W0}"] = early[BEFORE].mean(axis=1)
early[f"среднее {W1}"] = early[AFTER].mean(axis=1)
early["разница_пп"] = early[f"среднее {W1}"] - early[f"среднее {W0}"]
print("новых подписок:", len(new_subs), "| из них триалов:", int(new_subs["is_trial"].sum()))
early.round(2)

**Вывод.**

- Подписки, оформленные с сентября 2025 по июнь 2026, не заканчиваются раньше 30 дней: самая короткая жизнь ровно 30 дней, на первом продлении ушли 4 642 подписки.
  Значит, отмена срабатывает в конце оплаченного периода, и подписок "оформил и отменил в том же месяце" почти нет.
  Слепое пятно определения оттока на этих данных почти пустое.
- Доля новых подписок, которые ушли на первом продлении: **4,12% у оформленных в декабре-феврале и 4,15% у оформленных в марте-мае**,
  разница +0,03 п.п. По тарифам разница от -0,05 до +0,08 п.п. Триалов после повышения уходило даже меньше: 4,17% -> 3,69%.
- Те, кто подписался уже по новой цене, держатся так же, как подписавшиеся по старой.

**Итог раздела 4: гипотеза 2 не подтверждается.** После повышения цен отток, приток, отмены по дням, причины отмен,
неудачные платежи и ранний отток новых подписок остались на прежнем уровне. Единственный слабый сигнал: отток каналов
`social` и `performance` вырос на 0,1-0,2 п.п.

**Следующий шаг: графики.** Ответ на вопрос есть, теперь нужно 3-5 графиков, каждый из которых доказывает одну часть вывода.

## 5. Графики

Четыре графика, каждый доказывает одну часть вывода. Правила с Л6:

- **заголовок это вывод**, а не название ("выросла на 27%", а не "выручка по месяцам");
- **один акцентный цвет** (красный) для того, о чём вывод, остальное серым;
- у столбиков ось включает ноль; единицы подписаны на оси; источник мелко внизу;
- окна сравнения и порядок подстановок названы в подписи.

Картинки сохраняются в папку `project1/figures/` в корне репозитория: оттуда их берёт записка.

In [ ]:
from pathlib import Path

# Корень репозитория ищем по файлу pyproject.toml: так путь работает и из notebooks/, и из корня.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIG_DIR = ROOT / "project1" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ACCENT, GRAY, LIGHT = "#C1272D", "#6B7280", "#D1D5DB"    # акцентный красный, серый, светло-серый (Л6, слайд 23)
SOURCE = "Источник: prime.payments и prime.subscriptions; успешные платежи без двойных списаний; сутки по UTC."
MONTH_RU = {12: "дек", 1: "янв", 2: "фев", 3: "мар", 4: "апр", 5: "май", 6: "июн", 9: "сен", 10: "окт", 11: "ноя"}
plt.rcParams.update({"font.size": 11})


def ru(x: float, digits: int = 0) -> str:
    """Число по-русски: пробел в тысячах, запятая в дробной части."""
    return f"{x:,.{digits}f}".replace(",", " ").replace(".", ",")


def month_label(p: pd.Period) -> str:
    """Подпись месяца: "мар 26"."""
    return f"{MONTH_RU[p.month]} {str(p.year)[2:]}"


def finish(fig, ax, name: str, note: str = "", ydigits: int = 0) -> None:
    """Общее оформление: подписи оси Y по-русски, без лишних рамок, сетка по Y, источник внизу, сохранение в project1/figures."""
    ax.yaxis.set_major_formatter(lambda v, _: ru(v, ydigits))   # 1,5 вместо 1.5: как в подписях на графике
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB", zorder=0)
    fig.text(0.01, -0.02, (note + "\n" if note else "") + SOURCE, fontsize=8.5, color=GRAY, ha="left", va="top")
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")


print("графики сохраняем в:", FIG_DIR.relative_to(ROOT))

**Вывод.** Папка для картинок готова, общие правила оформления собраны в функции завершения графика (`finish`),
чтобы все четыре графика выглядели одинаково.

**Следующий шаг: график 1, средняя выручка в день по месяцам.** С него начинается вывод: что случилось и насколько.

In [ ]:
# График 1. Средняя выручка в день по месяцам: скачок в марте и новый уровень после.
pos = np.arange(len(m))                       # месяцы как позиции 0..6, так окна легко закрасить
y = m["revenue_per_day"] / 1e6                # млн руб. в день

fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
ax.axvspan(-0.5, 2.5, color=LIGHT, alpha=0.35, lw=0)     # окно "до": декабрь-февраль
ax.axvspan(2.5, 5.5, color=ACCENT, alpha=0.07, lw=0)     # окно "после": март-май
ax.plot(pos, y, marker="o", color=ACCENT, lw=2.5, zorder=3)
for i, v in zip(pos, y):
    ax.annotate(ru(v, 2), (i, v), textcoords="offset points", xytext=(0, 9), ha="center")
ax.text(1, 0.25, f"окно до\nв среднем {ru(rpd0 / 1e6, 2)} млн в день", ha="center", color=GRAY)
ax.text(4, 0.25, f"окно после\nв среднем {ru(rpd1 / 1e6, 2)} млн в день", ha="center", color=ACCENT)
ax.annotate("1 марта: Базовый 199 -> 249 руб.,\nРасширенный 399 -> 449 руб.", xy=(2.5, 1.2), xytext=(2.6, 0.75),
            color=GRAY, fontsize=9.5)
ax.axvline(2.5, color=GRAY, ls="--", lw=1)

ax.set_xticks(pos, [month_label(p) for p in m.index])
ax.set_ylim(0, y.max() * 1.15)                # ось от нуля: скачок видно честно, без растяжения
ax.set_ylabel("млн руб. в день")
ax.set_title(f"После повышения цен средняя выручка в день выросла на {ru(total_delta / rpd0 * 100)}% и держится на новом уровне",
             loc="left", fontsize=12.5)
finish(fig, ax, "1-vyruchka-v-den.png", "Окна: дек 2025 - фев 2026 и мар - май 2026, по 90 дней с данными. Апрель без 14-15.04 (дыра в выгрузке).",
       ydigits=1)
plt.show()

**Вывод.** График доказывает, **что случилось**: средняя выручка в день перешла на новый уровень ровно в марте
(1,55 -> 1,88 млн руб.) и дальше не откатилась (2,00 млн руб. в июне). В среднем за окно: 1,51 млн руб. в день до и 1,92 млн руб. после, +27%.
Ось начинается с нуля, поэтому скачок не преувеличен. Окна закрашены, чтобы было видно, какие месяцы сравниваем.

**Следующий шаг: график 2, каскад факторов.** Скачок виден, теперь нужно показать, из чего он сложился.

In [ ]:
# График 2. Каскад: из чего сложился рост средней выручки в день между окнами.
SHORT = {"active": "активные подписки", "per_sub_day": "платежей на подписку", "price": "цена платежа"}
steps2 = effects.rename(SHORT) / 1000          # тыс. руб. в день, порядок подстановок как в contrib

ax = waterfall(steps2, title=f"{ru(effects['price'] / total_delta * 100)}% роста средней выручки в день дала цена: "
                              f"+{ru(effects['price'] / 1000)} из +{ru(total_delta / 1000)} тыс. руб.")
fig = ax.figure
ax.set_ylabel("тыс. руб. в день")
# Перекрашиваем: цена акцентным цветом, остальное серым (в waterfall цвет по знаку, здесь вывод про цену).
for bar, name in zip(ax.patches, list(steps2.index) + ["итого"]):
    bar.set_color(ACCENT if name == "цена платежа" else (GRAY if name == "итого" else LIGHT))
finish(fig, ax, "2-kaskad-faktorov.png",
       f"Изменение средней выручки в день: {W0} -> {W1}. Порядок подстановок: активные подписки, платежи на подписку, цена.")
plt.show()

**Вывод.** График доказывает, **за счёт чего**: из +408 тыс. руб. в день цена платежа дала +296 тыс. (72%), рост базы +114 тыс.,
частота платежей на подписку почти ноль (-1 тыс.). Цена выделена акцентным цветом, потому что вывод про неё. Влияния показаны в рублях,
а не в долях (Л6, слайд 16). Порядок подстановок и окна названы в подписи.

**Следующий шаг: график 3, влияния по месяцам.** Каскад сравнивает два окна, но не показывает, что эффект цены разовый,
а рост базы постоянный. Это покажет помесячная разбивка.

In [ ]:
# График 3. Влияния по всем переходам месяц к месяцу: цена дала скачок один раз.
chart = all_steps.drop(index="всего").T[FACTORS] / 1000        # строки: переходы, столбцы: факторы, тыс. руб. в день
chart = chart.rename(columns=SHORT)
chart.index = [f"{month_label(pd.Period(a))} -> {month_label(pd.Period(b))}" for a, b in (s.split(" -> ") for s in chart.index)]

fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
chart.plot.bar(ax=ax, stacked=True, color=[GRAY, LIGHT, ACCENT], rot=0, width=0.65, zorder=2)
ax.axhline(0, color="#111827", lw=0.8)
ax.set_ylabel("тыс. руб. в день")
ax.legend(frameon=False, loc="upper left")
ax.annotate(f"цена: +{ru(chart.loc[chart.index[2], 'цена платежа'])}", xy=(2, chart.iloc[2].clip(lower=0).sum()),
            xytext=(0, 5), textcoords="offset points", ha="center", color=ACCENT)
ax.set_title("Цена дала скачок один раз, в марте; база добавляет 31-44 тыс. руб. в день каждый месяц", loc="left", fontsize=12.5)
finish(fig, ax, "3-vliyaniya-po-mesyacam.png", "Цепные подстановки для каждого перехода, порядок: активные подписки, платежи на подписку, цена.")
plt.show()

**Вывод.** График доказывает, что **эффект цены разовый и настоящий**: красный столбик есть только на переходе февраль -> март (+289 тыс. руб. в день),
на остальных переходах цены не менялись и красного почти нет. Серая часть, рост базы, одинаковая каждый месяц: и до повышения, и после.
Значит, повышение не сломало рост базы.

**Следующий шаг: график 4, отток.** Последний вопрос записки: не ушли ли клиенты из-за цены.

In [ ]:
# График 4. Отток за 30 дней по месяцам: общий (акцент) и каналы (серым).
pos4 = np.arange(len(CHURN_MONTHS))
overall = flow.loc["churn_30d_pct", list(CHURN_MONTHS)].astype(float)
by_ch = churn_by_channel[list(CHURN_MONTHS)]

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
labels = []                                                    # подписи у правого конца линий, без легенды
for ch_name, row in by_ch.iterrows():
    ax.plot(pos4, row.values, color=LIGHT, lw=1.3, zorder=2)
    labels.append([row.values[-1], ch_name, GRAY, "normal"])
ax.plot(pos4, overall.values, color=ACCENT, lw=3, marker="o", zorder=3)
labels.append([overall.values[-1], "все подписки", ACCENT, "bold"])

# Раздвигаем подписи, которые налезают друг на друга: идём снизу вверх, держим зазор не меньше 0,25 п.п.
labels.sort(key=lambda x: x[0])
for i in range(1, len(labels)):
    labels[i][0] = max(labels[i][0], labels[i - 1][0] + 0.25)
for y_lab, text_lab, color_lab, weight_lab in labels:
    ax.text(pos4[-1] + 0.15, y_lab, text_lab, va="center", fontsize=9, color=color_lab, fontweight=weight_lab)
ax.axvline(5.5, color=GRAY, ls="--", lw=1)                     # между февралём и мартом
ax.text(5.6, 0.4, "1 марта: повышение цен", color=GRAY, fontsize=9.5)

ax.set_xticks(pos4, [month_label(p) for p in CHURN_MONTHS])
ax.set_xlim(-0.3, pos4[-1] + 1.6)
ax.set_ylim(0, by_ch.values.max() * 1.15)                      # ось от нуля
ax.set_ylabel("отток за 30 дней, %")
b_mean, a_mean = flow.loc["churn_30d_pct", list(BEFORE)].mean(), flow.loc["churn_30d_pct", list(AFTER)].mean()
ax.set_title(f"Отток после повышения цен не вырос: {ru(b_mean, 2)}% до и {ru(a_mean, 2)}% после за 30 дней", loc="left", fontsize=12.5)
finish(fig, ax, "4-ottok.png",
       f"Отток: доля подписок, активных на 1-е число, закончившихся в месяце, в пересчёте на 30 дней.\n"
       f"Средние по окнам {W0} и {W1}. Серые линии: каналы привлечения.")
plt.show()

**Вывод.** График доказывает, что **отток не вырос**: красная линия (все подписки) лежит ровно на уровне 3,9% за 30 дней и до, и после 1 марта.
Каналы серым показывают обычный разброс: `organic` около 2,2%, `social` около 7%. У `social` после марта линия чуть выше,
но это десятые доли пункта, на фоне общего уровня почти не видно. Ось от нуля: так видно, что колебания маленькие.

**Следующий шаг: вывод для заказчика (раздел 6)** и записка: что случилось, за счёт чего, чего не знаем, что делать.

## 6. Вывод

*Заполним на следующих шагах.*